In [1]:
%load_ext autoreload
%autoreload 2

import os
os.chdir("../")

from ease_recommender import *
from npmi_recommender import *

import pickle as p

def create_mat(row, col, bool_to_int=True):
    # bool_to_int won't count duplicates in the same row, creates a different weighting basically
    if bool_to_int:
        data = np.ones_like(row, dtype=bool)
        return csr_matrix((data, (row, col))).astype(np.int64)
    else:
        data = np.ones_like(row, dtype=np.int64)
        return csr_matrix((data, (row, col)))

def check_if_all_terms_in_str(q, terms):
    for term in terms:
        if term not in q:
            return False

    return True

def get_cat2idx(category_type, D):
    if category_type == "track":
        return D["track2idx"]
    elif category_type == "album":
        return D["album2idx"]
    elif category_type == "artist":
        return D["artist2idx"]
    else:
        raise NotImplementedError

def find_match_using_terms(terms, cat2idx):
    matches = []
    for name in cat2idx.keys():
        if check_if_all_terms_in_str(name, terms):
            matches.append(name)

    if len(matches) > 1:
        raise Exception("Multiple matches found, filter down to a single match", matches)

    return matches[0]

print("loading cache data...")
D = p.load(open("cached_data/spotify_preprocessed.p", "rb"))

print("building csr matrices...")

# TODO: finish implementing track and album level recommendations

# track_mat = create_mat(D["playlist_indices"], D["track_indices"])
# album_mat = create_mat(D["playlist_indices"], D["album_indices"])
artist_mat = create_mat(D["playlist_indices"], D["artist_indices"])

print("done")

cat2idx = get_cat2idx("artist", D)
idx2cat = {v:k for k, v in cat2idx.items()}

loading cache data...
building csr matrices...
done


In [2]:
# use two items that you believe are similar to optimize the value of lambda_

a_name = find_match_using_terms(["sgeir", "7xUZ4069zcyBM4Bn10NQ1c"], cat2idx)
# a_name = find_match_using_terms(["7fNWySjsDn74LCawyJ27EQ"], cat2idx)

a = cat2idx[a_name]
print(f"{a_name=}")
print(f"Num Rows: {artist_mat[:, a].sum()}")

# a = cat2idx[find_match_using_terms(["Fleet Foxes"], cat2idx)]

# b = cat2idx[find_match_using_terms(["Fleet Foxes"], cat2idx)]
# b = cat2idx[find_match_using_terms(["Bon Iver", "4LEiUm1SRbFMgfqnQTwUbQ"], cat2idx)]
# b = cat2idx[find_match_using_terms(["SOHN"], cat2idx)]

# b_name = find_match_using_terms(["7fNWySjsDn74LCawyJ27EQ"], cat2idx)
b_name = find_match_using_terms(["Highas"], cat2idx)
b = cat2idx[b_name]
print(f"{b_name=}")
print(f"Num Rows: {artist_mat[:, b].sum()}")

c_name = find_match_using_terms(["Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)"], cat2idx)
c = cat2idx[c_name]
print(f"{c_name=}")
print(f"Num Rows: {artist_mat[:, c].sum()}")

d_name = find_match_using_terms(["Fleet Foxes"], cat2idx)
d = cat2idx[d_name]
print(f"{d_name=}")
print(f"Num Rows: {artist_mat[:, d].sum()}")

e_name = find_match_using_terms(["Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)"], cat2idx)
e = cat2idx[e_name]
print(f"{e_name=}")
print(f"Num Rows: {artist_mat[:, e].sum()}")

f_name = find_match_using_terms(["Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)"], cat2idx)
f = cat2idx[f_name]
print(f"{f_name=}")
print(f"Num Rows: {artist_mat[:, f].sum()}")

a_name='Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)'
Num Rows: 1981
b_name='Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)'
Num Rows: 412
c_name='Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)'
Num Rows: 71
d_name='Fleet Foxes (spotify:artist:4EVpmkEwrLYEg6jIsiPMIb)'
Num Rows: 12615
e_name='Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)'
Num Rows: 32356
f_name='Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)'
Num Rows: 64


In [3]:
mat = artist_mat
mat = csr_array(mat)
# mat = mat.tocoo()

In [4]:
n_users, n_items = mat.shape

X = mat.T @ mat
# X = X / n_users

X.shape

(295860, 295860)

In [5]:
X2 = X / n_users

In [6]:
X3 = X * csr_array(1/X.sum(axis=1)[:, None])

In [7]:
X4 = X * csr_array(1/X.sum(axis=1)[:, None])
X4 = X4 * csr_array(1/X4.sum(axis=1)[:, None])

In [9]:
# px = X.sum(axis=1)[:, None]
# py = X.sum(axis=0)[None, :]

# px = X.diagonal()
# px = px/px.sum()

# py = px[None, :]
# px = px[:, None]

In [10]:
# TODO: kind of an interesting way of calculating pmi sparsely, in theory we could add the alpha stuff here directly..?

In [11]:
# pmi = X2 * csr_array(1/px)
# pmi = pmi * csr_array(1/py)

In [12]:
import numpy as np
from scipy import sparse

def sparse_laplace_sppmi(X, alpha=1.0, normalize=False, zero_diag=True, non_neg=False):
    # Ensure input is CSR for fast row operations
    if not sparse.isspmatrix_csr(X):
        X = X.tocsr()
        
    # 1. Get Geometry of the Data
    # V: Vocabulary size (rows/cols)
    rows, cols = X.shape
    
    # N_raw: Total real observations
    N_raw = X.sum()
    
    # 2. Calculate "Smoothed" Marginals (The Global Statistics)
    # We pretend we added alpha to every cell, but we compute the sums analytically.
    
    # Virtual Total N = Real N + (alpha * Total Possible Cells)
    N_smoothed = N_raw + (alpha * rows * cols)
    
    # Raw marginals (sum of rows/cols)
    row_sums_raw = np.array(X.sum(axis=1)).flatten()
    col_sums_raw = np.array(X.sum(axis=0)).flatten()
    
    # Smoothed marginals = Raw Sum + (alpha * row_length)
    # Each row has 'cols' number of cells, so we add alpha * cols to the row sum
    P_x = (row_sums_raw + (alpha * cols)) / N_smoothed
    P_y = (col_sums_raw + (alpha * rows)) / N_smoothed
    
    # 3. Operate ONLY on Non-Zero Data (The Sparse Trick)
    # We extract the indices of existing data points to calculate their new PMI
    # efficiently, skipping the billions of zeros.
    
    # Create a copy to store results
    sppmi = X.copy().astype(np.float32)
    
    # Get indices of non-zero elements
    row_indices, col_indices = X.tocoo().nonzero()
    
    # Get the raw counts
    raw_counts = np.array(X.data)
    
    # Smooth the counts: Count_new = Count_raw + alpha
    smoothed_counts = raw_counts + alpha
    
    # Calculate P(x,y) for these specific entries
    P_xy = smoothed_counts / N_smoothed
    
    # 4. Vectorized PMI Calculation
    # PMI = log( P(x,y) / (P(x) * P(y)) )
    # Note: P_x[row_indices] grabs the specific P(x) for every non-zero entry
    
    # Numerator is P_xy
    # Denominator is P(x) * P(y)
    denominator = P_x[row_indices] * P_y[col_indices]
    
    # Calculate PMI (using log2)
    pmi_values = np.log2(P_xy / denominator)
    
    if normalize:
        pmi_values = pmi_values / -np.log2(P_xy)
    
    # 7. Update the matrix data
    sppmi.data = pmi_values
    
    if non_neg:
        sppmi.data = np.where(sppmi.data > 0, sppmi.data, 0)
    
    if zero_diag:
        sppmi.setdiag(np.zeros(sppmi.shape[0]))
    
    # 8. Clean up (Remove explicit zeros to keep matrix sparse)
    sppmi.eliminate_zeros()
    
    return sppmi

In [13]:
pmi2 = sparse_laplace_sppmi(X, alpha=.1 * .85, zero_diag=False)

In [14]:
pmi3 = pmi2.copy()
pmi3.data = 2**pmi3.data

In [48]:
pmi4 = pmi3 * csr_array(1/pmi3.sum(axis=1)[:, None])

In [83]:
pmi5 = pmi2 * csr_array(1/pmi2.sum(axis=1)[:, None])

In [89]:
from tqdm.auto import tqdm

import numpy as np
from scipy import sparse

n = n_items

# # --- Case A: "All" nodes (Global PageRank style) ---
# # Start with uniform probability
# v_all = np.ones(n) / n 
# # One iteration for EVERYONE
# v_next = v_all @ X 

# --- Case B: One Node of Interest (Node 42) ---
# Start with probability 1.0 at node 42
v_single = sparse.csr_array((1, n))
v_single[0, b] = 1.0

# mat_of_interest = pmi2.copy()
# mat_of_interest = pmi4.copy()

mat_of_interest = pmi4.copy()

# mat_of_interest = X.copy()
# mat_of_interest = X2.copy()
# mat_of_interest = X3.copy()
# mat_of_interest = X4.copy()

v_single_old = v_single.copy()
for _ in tqdm(range(2)):
    # v_single = v_single @ X3
    # v_single = v_single @ X2
    # v_single = (X2 @ v_single.T).T
    # v_single = (X3 @ v_single.T).T
    # v_single = (X3.T @ v_single.T).T
    # v_single = v_single @ X3.T
    
    v_single = v_single @ mat_of_interest
    
    err = np.abs(v_single - v_single_old).max()
    print("for convergence:", err)

    if err <= 1e-6:
        break
    
    v_single_old = v_single
#     mat_of_interest.data = mat_of_interest.data * (1/5250) 
#     mat_of_interest.data = mat_of_interest.data * 0 + 1
    mat_of_interest.data = mat_of_interest.data * 1
    
    scores = v_single.toarray()[0]
    print("target metric:", np.argsort(-scores).tolist().index(c))
    
# scores = v_single.toarray()[0]

# np.argsort(-scores).tolist().index(c)

  0%|          | 0/2 [00:00<?, ?it/s]

for convergence: 0.9741554670022656
target metric: 1
for convergence: 0.02584453299773438
target metric: 1


In [82]:
for idx in np.argsort(-scores)[:20]:
    print(idx2cat[idx])

Drake (spotify:artist:3TVXtAsR1Inumwj472S9r4)
Rihanna (spotify:artist:5pKCCKE2ajJHZ9KAiaK11H)
The Chainsmokers (spotify:artist:69GGBxA162lTqCwzJG5jLp)
The Weeknd (spotify:artist:1Xyo4u8uXC1ZmMpatF05PJ)
Ed Sheeran (spotify:artist:6eUKZXaKkcviH0Ku9w2n3V)
Calvin Harris (spotify:artist:7CajNmpbOovFoOoasH2HaY)
Kanye West (spotify:artist:5K4W6rqBFWDnAN6FQUkS6x)
Maroon 5 (spotify:artist:04gDigrS5kc9YWfZHwBETP)
Coldplay (spotify:artist:4gzpq5DPGxSnKTe4SA8HAU)
Major Lazer (spotify:artist:738wLrAtLtCtFOLvQBXOXp)
Justin Bieber (spotify:artist:1uNFoZAHBGtllmzznpCI3s)
Imagine Dragons (spotify:artist:53XhwfbYqKCa1cC15pYq2q)
Kendrick Lamar (spotify:artist:2YZyLoL8N0Wb9xBt1NhZWg)
Beyoncé (spotify:artist:6vWDO969PvNqNYHIOW5v0m)
Bruno Mars (spotify:artist:0du5cEVh5yTK9QJze8zA0C)
Sia (spotify:artist:5WUlDfRSoLAfcVSX1WnrxN)
Twenty One Pilots (spotify:artist:3YQKmKGau1PzlVlkL1iodx)
Kygo (spotify:artist:23fqKkggKUBHNkbKtXEls4)
Ariana Grande (spotify:artist:66CXWjxzNUsdJxJ2JdwvnR)
Jason Derulo (spotify:artis

In [134]:
scores = pmi2[b].toarray()

for idx in np.argsort(-scores)[:20]:
    print(idx2cat[idx])

Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)
Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)
Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)
Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)
Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)
Susanne Sundfør (spotify:artist:54KCNI7URCrG6yjQK3Ukow)
Ings (spotify:artist:3wJ2NeHF58Im2tojNX8ESR)
Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)
Alice Boman (spotify:artist:3WiytRnvoL0kT3oAGl9TCt)
Karpe Diem (spotify:artist:3X23gpg1vPacr0hBARyxtN)
Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
Hanzee (spotify:artist:5yM1po4NHvE2yE1Kf84LWJ)
ZL-Project (spotify:artist:4w8PGLhS3yzYSzeV3x2hkA)
Majical Cloudz (spotify:artist:4BEYBN6NCPrFk3sOLMTby3)
The White Birch (spotify:artist:3pDnwHQUEXbkNX3cvidKpG)
Amason (spotify:artist:4cJKxS7uOPhwb5UQ70sYpN)
SOAK (spotify:artist:4PLsMEk2DCRVlVL2a9aZAv)
Hkeem (spotify:artist:46XcyK8FnyCJJlvYCUwVZH)
Oh Pep! (spotify:artist:3L9rqEIsNSaOcx2QIstn7v)
Farao (spotify:artist:6XIX2G6ZGiVQg

In [ ]:
from tqdm.auto import tqdm

import numpy as np
from scipy import sparse

n = n_items

# # --- Case A: "All" nodes (Global PageRank style) ---
# # Start with uniform probability
# v_all = np.ones(n) / n 
# # One iteration for EVERYONE
# v_next = v_all @ X 

# --- Case B: One Node of Interest (Node 42) ---
# Start with probability 1.0 at node 42
v_single = sparse.csr_array((1, n))
v_single[0, b] = 1.0

v_single_old = v_single.copy()
for _ in tqdm(range(1)):
    # v_single = v_single @ X3
    # v_single = v_single @ X2
    # v_single = (X2 @ v_single.T).T
    # v_single = (X3 @ v_single.T).T
    # v_single = (X3.T @ v_single.T).T
    # v_single = v_single @ X3.T

    v_single = v_single @ X4
    
    err = np.abs(v_single - v_single_old).max()
    print(err)

    if err <= 1e-6:
        break
    
    v_single_old = v_single

scores = v_single.toarray()[0]

np.argsort(-scores).tolist().index(c)

In [97]:
for idx in np.argsort(-scores)[:20]:
    print(idx2cat[idx])

Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)
Hozier (spotify:artist:2FXC3k01G6Gw61bmprjgqS)
alt-J (spotify:artist:3XHO7cRUPCLOr6jwp8vsx5)
Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)
Coldplay (spotify:artist:4gzpq5DPGxSnKTe4SA8HAU)
Sia (spotify:artist:5WUlDfRSoLAfcVSX1WnrxN)
Ed Sheeran (spotify:artist:6eUKZXaKkcviH0Ku9w2n3V)
Sylvan Esso (spotify:artist:39vA9YljbnOApXKniLWBZv)
Glass Animals (spotify:artist:4yvcSjfu4PC0CYQyLy4wSq)
Lana Del Rey (spotify:artist:00FQb4jTyendYWaN8pK0wa)
Tame Impala (spotify:artist:5INjqkS1o8h1imAzPqGZBb)
Broods (spotify:artist:5r5Va4lVQ1zjEfbJSrmCsS)
Vance Joy (spotify:artist:10exVja0key0uqUkk6LJRT)
Lorde (spotify:artist:163tK9Wjr9P9DmM0AVK7lm)
Birdy (spotify:artist:2WX2uTcsvV5OnS0inACecP)
M83 (spotify:artist:63MQldklfxkjYDoUE4Tppz)
The Lumineers (spotify:artist:16oZKvXb6WkQlVAjwo2Wbg)
The Weeknd (spotify:artist:1Xyo4u8uXC1ZmMpatF05PJ)
Lord Huron (spotify:artist:6ltzsmQQbmdoHHbLZ4ZN25)
James Blake (spotify:artist:53KwLdlmrlCelAZMaLVZqU)


In [21]:
# x2 = X3 @ v_single.T

In [22]:
# x2.toarray()

In [23]:
# x.toarray()

In [8]:
v_single.shape

(1, 1000000)

In [5]:
# %%time

# import rustworkx as rx

# graph = rx.PyDiGraph()
    
# graph.add_nodes_from(range(mat.shape[0]))

# rows, cols = mat.tocoo().nonzero()

# graph.add_edges_from_no_data(zip(rows, cols))

# import gc

# del rows, cols
# gc.collect()

In [18]:
import numpy as np
import rustworkx as rx
from scipy import sparse

def get_similar_items(interaction_matrix, target_item_idx, alpha=0.85):
    """
    Calculates item similarity using a Random Walk (Personalized PageRank) on a bipartite graph.
    
    Args:
        interaction_matrix (csr_array): Boolean sparse matrix (n_users x n_items).
        target_item_idx (int): The original column index of the item to query.
        top_k (int): Number of similar items to return.
        alpha (float): Restart probability (1 - alpha is the prob of jumping back to seed).
                       Higher alpha = longer walks (global structure).
                       Lower alpha = shorter walks (local structure).
    
    Returns:
        List[scan]: List of tuples (item_index, similarity_score).
    """
    
    n_users, n_items = interaction_matrix.shape
    
    # 1. Coordinate Extraction
    # Efficiently get row (user) and col (item) indices from CSR
    user_indices, item_indices = interaction_matrix.nonzero()
    
    # 2. Index Offsetting
    # We shift item indices up by n_users so they have unique IDs in the graph
    # Nodes 0 to (n_users-1) are Users
    # Nodes n_users to (n_users + n_items - 1) are Items
    item_nodes = item_indices + n_users
    
    # 3. Build Edge List (Bidirectional)
    # We need User -> Item AND Item -> User for the walk to traverse back and forth
    # Stack the edges: (User, Item) and (Item, User)
    sources = np.concatenate([user_indices, item_nodes])
    targets = np.concatenate([item_nodes, user_indices])
    
    # 4. Create Graph
    # PyDiGraph is directed. We use from_edge_list for speed.
    graph = rx.PyDiGraph()
    # We explicitly add nodes to ensure disconnected nodes are accounted for if needed
    graph.add_nodes_from(range(n_users + n_items))
    graph.add_edges_from_no_data(zip(sources, targets))
    
    # 5. Define the Seed Node
    # The graph node ID for our target item
    seed_node = target_item_idx + n_users
    
    # 6. Run Personalized PageRank (Random Walk with Restart)
    # This calculates the stationary distribution of a random walker starting at seed_node
    # personalization={node_id: score} defines the restart distribution
    ppr_scores = rx.pagerank(
        graph, 
        alpha=alpha, 
        personalization={seed_node: seed_node}
    )
    
    # 7. Filter and Sort Results
    # We only care about scores for Item nodes (indices >= n_users)
    # We also exclude the target item itself
    item_scores = []
    
    for node_id, score in ppr_scores.items():
        # Check if this node is an Item node
        if node_id >= n_users:
            original_item_idx = node_id - n_users
            if original_item_idx != target_item_idx:
                item_scores.append((original_item_idx, score))
    
    # Sort by score descending
    # item_scores.sort(key=lambda x: x[1], reverse=True)
    item_scores.sort(key=lambda x: x[0])

    item_ids, scores = zip(*item_scores)

    return np.array(scores)

In [25]:
%%time

alpha = 1/2

b_scores = get_similar_items(mat, b, alpha)
# c_scores = get_similar_items(mat, c, alpha)

CPU times: total: 26.7 s
Wall time: 26.8 s


In [26]:
np.argsort(-b_scores).tolist().index(c)

54179

In [6]:
# import rustworkx as rx
# import numpy as np

# def compute_rwr_rustworkx(adj_matrix, seed_index, restart_prob=0.15):
#     """
#     Computes RWR using Rustworkx.
#     """
#     # 1. Efficiently convert Scipy CSR to Rustworkx Graph
#     # This is the most expensive step. Do this once if possible.
#     # We utilize the instance method for directed graphs
#     graph = rx.PyDiGraph.from_adjacency_matrix(adj_matrix)
    
#     # 2. Run PageRank with Personalization (Equivalent to RWR)
#     # personalization={seed_node: 1.0} forces the restart to always go to the seed
#     # alpha is the damping factor (1 - restart_prob)
#     scores = rx.pagerank(
#         graph,
#         alpha=1.0 - restart_prob,
#         personalization={seed_index: 1.0},
#         tol=1e-6
#     )
    
#     # Rustworkx returns a dictionary {node_index: score}
#     # Convert back to array if needed
#     rwr_vector = np.zeros(adj_matrix.shape[0])
#     for node, score in scores.items():
#         rwr_vector[node] = score
        
#     return rwr_vector

In [7]:
import rustworkx as rx
import numpy as np

def compute_rwr_rustworkx(graph, restart_prob):
    """
    Computes RWR using Rustworkx.
    """
    
    assert restart_prob >= 0
    assert restart_prob <= 1

    
    # 2. Run PageRank with Personalization (Equivalent to RWR)
    # personalization={seed_node: 1.0} forces the restart to always go to the seed
    # alpha is the damping factor (1 - restart_prob)
    scores = rx.pagerank(
        graph,
        alpha=1.0 - restart_prob,
        # personalization={seed_index: 1.0},
        tol=1e-6
    )
    
    # Rustworkx returns a dictionary {node_index: score}
    # Convert back to array if needed
    rwr_vector = np.zeros(graph.num_nodes())
    for node, score in scores.items():
        rwr_vector[node] = score
        
    return rwr_vector

In [22]:
%%time

graph = rx.PyDiGraph()
    
graph.add_nodes_from(range(mat.shape[0]))

rows, cols = mat.tocoo().nonzero()

graph.add_edges_from_no_data(zip(rows, cols))

import gc

del rows, cols
gc.collect()

CPU times: total: 5.89 s
Wall time: 5.93 s


228

In [23]:
def get_metric(i, j, Y):
    return np.argsort(-Y[i].toarray()).tolist().index(j)

In [24]:
rwr = compute_rwr_rustworkx(graph, .15)

In [28]:
from scipy import sparse

def compute_rwr_scipy(adj_matrix, seed_index=None, restart_prob=0.15, max_iter=100, tol=1e-6):
    """
    Computes Random Walk with Restart (RWR) using direct Scipy sparse matrix operations.
    
    Args:
        adj_matrix (scipy.sparse.csr_array): Boolean or weighted adjacency matrix.
        seed_index (int): The index of the node to restart to (seed node).
        restart_prob (float): Probability of restarting (0 < r < 1).
        max_iter (int): Maximum iterations.
        tol (float): Convergence tolerance.
        
    Returns:
        np.array: The RWR score vector.
    """
    # 1. Normalize the adjacency matrix to create a Transition Matrix (M)
    # Convert to float to handle division
    adj_matrix = adj_matrix.astype(float)
    
    # Calculate row sums (out-degree)
    row_sums = np.array(adj_matrix.sum(axis=1)).flatten()
    
    # Handle sink nodes (nodes with 0 out-degree) to prevent division by zero
    # We effectively add a self-loop to sink nodes or restart from them
    row_sums[row_sums == 0] = 1.0 
    
    # Create the diagonal normalization matrix D^-1
    inv_d = sparse.diags(1.0 / row_sums)
    
    # M = D^-1 * A (Row-stochastic transition matrix)
    # Note: Use A * D^-1 for column-stochastic if your math expects that
    M = inv_d @ adj_matrix 
    
    # 2. Initialize the RWR vector
    n = adj_matrix.shape[0]
    r = np.zeros(n)
    if seed_index is not None:
        r[seed_index] = 1.0  # Start at seed
    
    # Restart vector (one-hot vector for the seed)
    q = np.zeros(n)
    if seed_index is not None:
        q[seed_index] = 1.0
    
    # 3. Power Iteration: r_new = (1 - c) * M^T * r + c * q
    # Note: We transpose M because RWR is usually formulated as v = M^T v
    # If your adjacency is source->target, M.T moves probability correctly.
    M_T = M.T
    
    for _ in range(max_iter):
        r_new = (1 - restart_prob) * (M_T @ r) + (restart_prob * q)
        
        # Check convergence (L1 norm)
        if np.linalg.norm(r_new - r, 1) < tol:
            return r_new
        r = r_new
        
    return r

In [29]:
%%time

r = compute_rwr_scipy(mat)

ValueError: operands could not be broadcast together with shapes (295860,) (1000000,) 

In [17]:
len(rwr)

1000000

In [14]:
rwr.shape

(1000000,)

In [12]:
Y = rwr

metrics = []
for i in [a, b, c, d, e, f]:
    for j in [a, b, c, d, e, f]:
        if i != j:
            metrics.append(get_metric(i, j, Y))
    
np.mean(metrics)

AttributeError: 'numpy.float64' object has no attribute 'toarray'

In [6]:
# import pandas as pd

# for corr_type in ["pearson", "spearman", "kendall"]:
#     print(round(100*pd.DataFrame({
#         "a": Y[b].toarray(),
#         "b": Y[c].toarray(),
#     }).corr(corr_type)["a"]["b"], 1))

# metrics = [
# #     get_metric(a, b, Y),
# #     get_metric(b, a, Y),
    
#     get_metric(b, c, Y),
#     get_metric(c, b, Y),
# ]

# np.mean(metrics)